# 📉 Topic 06 — Data Drift & Unsupervised Anomaly Detection

Welcome! In this notebook, we will learn how to protect our machine learning models once they are out in the wild.

We will build **Smoke Alarms** that alert us when:
1. The entire dataset starts to slowly shift away from what we expect (**Data Drift**).
2. A single, crazy, impossible data point enters the system (**Anomaly Detection**).

## 🕰️ 1. Simulating Data Drift

Let's imagine we are a bank. We trained a model in **January (Reference Data)**. Now it is **July (Current Data)**. 

Due to an economic event, customer incomes have shifted. Let's create this fake data and visualize it.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

np.random.seed(42)

# 1. Create January Data (Reference)
# Average income is $50k
jan_income = np.random.normal(loc=50000, scale=10000, size=1000)

# 2. Create July Data (Current) - DRIFTED!
# A new marketing campaign brought in wealthier clients. Average income is now $70k!
jul_income = np.random.normal(loc=70000, scale=12000, size=1000)

# 3. Plot them together using a Kernel Density Estimate (KDE) plot
plt.figure(figsize=(10, 5))
sns.kdeplot(jan_income, fill=True, color="#4A90D9", label="January (Reference)")
sns.kdeplot(jul_income, fill=True, color="#E94B4B", label="July (Current/Drifted)")
plt.title("Visualizing Feature Drift in Customer Income", fontsize=16)
plt.xlabel("Annual Income ($)")
plt.ylabel("Density")
plt.legend()
plt.show()

## 🚨 2. Catching Drift with Math (PSI)

Looking at charts is nice, but we can't look at charts for 500 features every single day. We need a number! 

**Population Stability Index (PSI)** is a mathematical score:
- **< 0.1** = Safe. No drift.
- **0.1 to 0.2** = Warning. Keep an eye on it.
- **> 0.2** = CRITICAL ALARM! Data has drifted heavily. Model is in danger.

Let's write a function to calculate it.

In [ ]:
def calculate_psi(expected, actual, buckets=10):
    """Calculates the Population Stability Index (PSI)."""
    # Break data into buckets (bins)
    breakpoints = np.arange(0, buckets + 1) / buckets * 100
    breakpoints = np.percentile(expected, breakpoints)
    
    # Count how many people fall into each bucket
    expected_percents = np.histogram(expected, breakpoints)[0] / len(expected)
    actual_percents = np.histogram(actual, breakpoints)[0] / len(actual)
    
    # Replace 0s with a tiny number to avoid math errors (dividing by zero)
    expected_percents = np.clip(expected_percents, a_min=0.0001, a_max=None)
    actual_percents = np.clip(actual_percents, a_min=0.0001, a_max=None)
    
    # THE PSI FORMULA
    psi_value = np.sum((actual_percents - expected_percents) * np.log(actual_percents / expected_percents))
    return psi_value

psi_score = calculate_psi(jan_income, jul_income)

print(f"The PSI Score for Income is: {psi_score:.3f}")
if psi_score > 0.2:
    print("\u26a0\ufe0f ALARM! Major Drift Detected! (> 0.2)")
elif psi_score > 0.1:
    print("\ud83d\udc40 WARNING! Slight Drift Detected! (0.1 - 0.2)")
else:
    print("\u2705 SAFE! No significant drift detected. (< 0.1)")

## 🌲 3. Catching Anomalies with Isolation Forest

Drift happens to the whole dataset. But what if just **one** customer enters an income of $999,999,999 due to a typo? We need **Anomaly Detection**.

**Isolation Forest** is an algorithm that draws random boxes around data. Weird data points are easier to box in (isolate) than normal ones.

Let's create some normal customers, and inject 5% crazy anomalies!

In [ ]:
from sklearn.ensemble import IsolationForest

# 1. Create normal data (Income and Age)
normal_data = np.random.randn(500, 2) * 2 + 20

# 2. Create 25 crazy anomalies (very high or very low numbers)
anomalies = np.random.uniform(low=-10, high=50, size=(25, 2))

# Combine them
X_production = np.vstack([normal_data, anomalies])

# 3. Train the Isolation Forest
iso_forest = IsolationForest(contamination=0.05, random_state=42) # We expect ~5% anomalies
iso_forest.fit(X_production)

# 4. Predict! 
# Isolation Forest returns 1 for Normal, and -1 for Anomaly.
predictions = iso_forest.predict(X_production)

# Let's visualize how well it caught the anomalies!
plt.figure(figsize=(10, 6))
# Plot Normal points (where prediction == 1)
plt.scatter(X_production[predictions == 1, 0], X_production[predictions == 1, 1], c='#4A90D9', label='Normal (1)', alpha=0.6)
# Plot Anomalies (where prediction == -1)
plt.scatter(X_production[predictions == -1, 0], X_production[predictions == -1, 1], c='#E94B4B', label='Anomaly (-1)', marker='x', s=100)

plt.title("Isolation Forest at Work: Catching the Outliers", fontsize=16)
plt.legend()
plt.show()

## 📝 Key Takeaways & Quiz

### Key Takeaways:
- **Models Decay:** When the real world changes, your model's accuracy drops. You must monitor it!
- **PSI (Population Stability Index):** The standard metric for Drift. Above 0.2 means retrain your model.
- **Isolation Forest:** A powerful algorithm that finds weird data points (anomalies) without needing you to tell it what an anomaly looks like.

### ❓ Quiz (Check your understanding)
1. A global pandemic changes how people spend money overnight. Is this Feature Drift or Concept Drift?
2. If your PSI score for `age` is 0.05, should you be worried?
3. Why is an Isolation Forest considered an "unsupervised" algorithm?